In [1]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

recipes = pd.read_csv(
    "../../datasets/RAW_recipes.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)
print("Recipes shape:", recipes.shape)

Train shape: (428067, 5)
Test shape: (106980, 5)
Recipes shape: (231637, 12)


In [2]:
content_columns = [
    "id",
    "name",
    "tags",
    "description",
    "ingredients"
]

recipes[content_columns].head()

,id,name,tags,description,ingredients
0,137739,arriba baked winter squash mexican style,"['60-minutes-or-less', 'time-to-make', 'course...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ..."
1,31490,a bit different breakfast pizza,"['30-minutes-or-less', 'time-to-make', 'course...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg..."
2,112140,all in the kitchen chili,"['time-to-make', 'course', 'preparation', 'mai...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato..."
3,59389,alouette potatoes,"['60-minutes-or-less', 'time-to-make', 'course...","this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n..."
4,44061,amish tomato ketchup for canning,"['weeknight', 'time-to-make', 'course', 'main-...",my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar..."


In [3]:
# Popunimo eventualno nedostajuće vrednosti
recipes["tags"] = recipes["tags"].fillna("")
recipes["description"] = recipes["description"].fillna("")
recipes["ingredients"] = recipes["ingredients"].fillna("")

# Spojimo sadržaj recepta u jedan tekst
recipes["content"] = (
        recipes["tags"] + " " +
        recipes["ingredients"] + " " +
        recipes["description"]
)

print("Number of recipes:", len(recipes))
print("Example content:")
print(recipes.loc[0, "content"])

Number of recipes: 231637
Example content:
['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event', 'vegetarian', 'winter', 'dietary', 'christmas', 'seasonal', 'squash'] ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt'] autumn is my favorite time of year to cook! this recipe 
can be prepared either spicy or sweet, your choice!
two of my posted mexican-inspired seasoning mix recipes are offered as suggestions.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(recipes["content"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of stored values:", tfidf_matrix.nnz)

TF-IDF matrix shape: (231637, 50000)
Number of stored values: 11728282


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Izaberi jedan recept za testiranje
test_recipe_index = 0

# Sličnost ovog recepta sa svim ostalim receptima
similarities = cosine_similarity(
    tfidf_matrix[test_recipe_index],
    tfidf_matrix
).flatten()

# Izbaci sam recept
similarities[test_recipe_index] = -1

# Indeksi 10 najsličnijih recepata
similar_recipe_indices = np.argsort(similarities)[-10:][::-1]

# Prikaži rezultate
similar_recipes = recipes.iloc[similar_recipe_indices][
    ["id", "name"]
].copy()

similar_recipes["similarity"] = similarities[similar_recipe_indices]

similar_recipes

,id,name,similarity
149516,76838,orange winter squash casserole,0.412056
15510,254360,baked winter squash,0.378530
130241,189352,maple sweet dumpling squash,0.365378
92696,254407,glazed butternut squash,0.353933
229206,31402,yellow squash dressing,0.348168
40860,168161,cheesy summer squash spears,0.346695
104852,145462,herb roasted winter squash,0.346348
32535,379235,butternut squash with browned butter and thyme,0.342764
230623,430958,zesty zucchini,0.340701
176256,228436,roasted summer squash boats,0.325139


In [6]:
test_user = test_ratings["user_id"].iloc[0]

user_train_ratings = train_ratings[
    train_ratings["user_id"] == test_user
    ].copy()

print("Test user:", test_user)
print("Number of training ratings:", len(user_train_ratings))

user_train_ratings

Test user: 162826
Number of training ratings: 323


,user_id,recipe_id,date,rating,review
6960,162826,220314,2007-05-26,5,Very tasty! When I make again; think I'll try ...
9384,162826,314978,2010-02-26,5,"I wish for 10 stars for this recipe. Bayhill, ..."
10877,162826,5229,2006-04-16,5,Delicious!!!! Used homemade Reuben Bread. MMMMM!
12009,162826,22719,2008-07-24,5,"These are delicious! Lots of possible add-ins,..."
12211,162826,208119,2007-11-10,5,"Yum Yum, Sharon. Love these lettle jewels. Aft..."
...,...,...,...,...,...
423325,162826,27931,2005-05-03,5,"Not only great sandwich, but a fun one. Thank ..."
427020,162826,72801,2007-05-09,5,"AAA...cheesy omelet, Sharon. Made for A TASTE ..."
427031,162826,117464,2008-07-05,5,"This is a AAA burger, Becky!!! I had forgotten..."
427826,162826,9400,2008-02-04,4,"A good, dense meatloaf. Next time, will reduce..."


In [7]:
# Recepti koje je korisnik ocenio
rated_recipe_ids = user_train_ratings["recipe_id"].values

# Pronađi njihove indekse u recipes tabeli
recipe_indices = recipes.index[
    recipes["id"].isin(rated_recipe_ids)
]

# TF-IDF vektori ocenjenih recepata
user_recipe_vectors = tfidf_matrix[recipe_indices]

# Ocene korisnika za te recepte
user_ratings = user_train_ratings[
    user_train_ratings["recipe_id"].isin(
        recipes.iloc[recipe_indices]["id"]
    )
]["rating"].values

print("Rated recipes found:", user_recipe_vectors.shape[0])
print("Ratings used:", len(user_ratings))

Rated recipes found: 323
Ratings used: 323


In [8]:
# Pretvaramo ocene u težine
weights = user_ratings.astype(float)

# Ponderisani profil korisnika
user_profile = user_recipe_vectors.multiply(
    weights.reshape(-1, 1)
).sum(axis=0)

# Normalizujemo profil
user_profile = user_profile / weights.sum()

print("User profile shape:", user_profile.shape)

User profile shape: (1, 50000)


In [10]:
# Pretvaramo user profile iz np.matrix u običan numpy array
user_profile = np.asarray(user_profile)

# Proveravamo oblik
print("User profile shape:", user_profile.shape)

# Sličnost user profila sa svim receptima
user_similarities = cosine_similarity(
    user_profile,
    tfidf_matrix
).flatten()

# Ne preporučujemo recepte koje je korisnik već ocenio
rated_indices = recipes.index[
    recipes["id"].isin(rated_recipe_ids)
]

user_similarities[rated_indices] = -1

# Top 10 preporuka
top_indices = np.argsort(user_similarities)[-10:][::-1]

content_recommendations = recipes.iloc[top_indices][
    ["id", "name"]
].copy()

content_recommendations["score"] = user_similarities[top_indices]

content_recommendations

User profile shape: (1, 50000)


,id,name,score
147959,128975,one dish meal,0.603865
87569,172637,fried chicken breast without breading,0.591047
15415,56376,baked tomato,0.585640
152999,35087,parmesan onion bake,0.564931
142531,79281,najwa s chicken soup,0.560127
230873,215178,zucchini tomato skillet,0.556225
42392,76222,chicken stuffed bell peppers,0.555041
79630,136241,eggplant tomato deluxe,0.555032
124834,35130,lion knees potatoes,0.553729
230901,38677,zucchini and caramelized onions,0.553378


In [11]:
def recommend_content_based(user_id, k=10):
    # Ocene korisnika iz trening skupa
    user_ratings = train_ratings[
        train_ratings["user_id"] == user_id
        ].copy()

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Recepti koje je korisnik ocenio
    rated_recipe_ids = user_ratings["recipe_id"].values

    # Pronađi njihove indekse u recipes tabeli
    recipe_indices = recipes.index[
        recipes["id"].isin(rated_recipe_ids)
    ]

    if len(recipe_indices) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # TF-IDF vektori ocenjenih recepata
    user_recipe_vectors = tfidf_matrix[recipe_indices]

    # Ocene korisnika
    valid_ratings = user_ratings[
        user_ratings["recipe_id"].isin(
            recipes.iloc[recipe_indices]["id"]
        )
    ]["rating"].values.astype(float)

    # User profile
    profile = user_recipe_vectors.multiply(
        valid_ratings.reshape(-1, 1)
    ).sum(axis=0)

    profile = profile / valid_ratings.sum()
    profile = np.asarray(profile)

    # Sličnost sa svim receptima
    similarities = cosine_similarity(
        profile,
        tfidf_matrix
    ).flatten()

    # Ne preporučujemo već ocenjene recepte
    rated_indices = recipes.index[
        recipes["id"].isin(rated_recipe_ids)
    ]

    similarities[rated_indices] = -1

    # Top-k
    top_indices = np.argsort(similarities)[-k:][::-1]

    recommendations = recipes.iloc[top_indices][
        ["id", "name"]
    ].copy()

    recommendations["score"] = similarities[top_indices]

    return recommendations

In [12]:
recommend_content_based(162826, k=10)

,id,name,score
147959,128975,one dish meal,0.603865
87569,172637,fried chicken breast without breading,0.591047
15415,56376,baked tomato,0.585640
152999,35087,parmesan onion bake,0.564931
142531,79281,najwa s chicken soup,0.560127
230873,215178,zucchini tomato skillet,0.556225
42392,76222,chicken stuffed bell peppers,0.555041
79630,136241,eggplant tomato deluxe,0.555032
124834,35130,lion knees potatoes,0.553729
230901,38677,zucchini and caramelized onions,0.553378


In [13]:
content_recommendations = recommend_content_based(
    user_id=162826,
    k=10
)

content_recommendations

,id,name,score
147959,128975,one dish meal,0.603865
87569,172637,fried chicken breast without breading,0.591047
15415,56376,baked tomato,0.585640
152999,35087,parmesan onion bake,0.564931
142531,79281,najwa s chicken soup,0.560127
230873,215178,zucchini tomato skillet,0.556225
42392,76222,chicken stuffed bell peppers,0.555041
79630,136241,eggplant tomato deluxe,0.555032
124834,35130,lion knees potatoes,0.553729
230901,38677,zucchini and caramelized onions,0.553378


In [14]:
def evaluate_content_based(k=10, max_users=1000):
    users = test_ratings["user_id"].unique()[:max_users]

    precisions = []

    for user_id in users:
        recommendations = recommend_content_based(
            user_id=user_id,
            k=k
        )

        if len(recommendations) == 0:
            continue

        recommended_ids = set(recommendations["id"])

        relevant_ids = set(
            test_ratings[
                (test_ratings["user_id"] == user_id) &
                (test_ratings["rating"] >= 4)
                ]["recipe_id"]
        )

        if len(relevant_ids) == 0:
            continue

        hits = len(recommended_ids & relevant_ids)

        precisions.append(hits / k)

    return np.mean(precisions)

In [15]:
content_precision_at_10 = evaluate_content_based(
    k=10,
    max_users=1000
)

print(
    "Content-Based Precision@10:",
    content_precision_at_10
)

Content-Based Precision@10: 0.0006006006006006007


In [16]:
recipe_id_to_index = pd.Series(
    recipes.index,
    index=recipes["id"]
)

print("Recipe ID mapping created:", len(recipe_id_to_index))

Recipe ID mapping created: 231637


In [17]:
def recommend_content_based(user_id, k=10):
    user_ratings = train_ratings[
        train_ratings["user_id"] == user_id
        ].copy()

    # Koristimo samo pozitivne ocene
    user_ratings = user_ratings[
        user_ratings["rating"] >= 4
        ]

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Zadržavamo samo recepte koji postoje u recipes tabeli
    user_ratings = user_ratings[
        user_ratings["recipe_id"].isin(recipe_id_to_index.index)
    ]

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Indeksi TF-IDF vektora - U ISTOM REDOSLEDU kao ocene
    recipe_indices = user_ratings["recipe_id"].map(
        recipe_id_to_index
    ).values

    user_recipe_vectors = tfidf_matrix[recipe_indices]

    # Ocene su sada pravilno poravnate sa vektorima
    weights = user_ratings["rating"].values.astype(float)

    # User profile
    user_profile = user_recipe_vectors.multiply(
        weights.reshape(-1, 1)
    ).sum(axis=0)

    user_profile = user_profile / weights.sum()
    user_profile = np.asarray(user_profile)

    # Sličnost sa svim receptima
    similarities = cosine_similarity(
        user_profile,
        tfidf_matrix
    ).flatten()

    # Ne preporučujemo već ocenjene recepte
    rated_recipe_ids = user_ratings["recipe_id"].values

    rated_indices = [
        recipe_id_to_index[recipe_id]
        for recipe_id in rated_recipe_ids
    ]

    similarities[rated_indices] = -1

    # Top-k preporuka
    top_indices = np.argsort(similarities)[-k:][::-1]

    recommendations = recipes.iloc[top_indices][
        ["id", "name"]
    ].copy()

    recommendations["score"] = similarities[top_indices]

    return recommendations

In [18]:
content_recommendations = recommend_content_based(
    user_id=162826,
    k=10
)

content_recommendations

,id,name,score
147959,128975,one dish meal,0.603736
87569,172637,fried chicken breast without breading,0.589297
15415,56376,baked tomato,0.586969
152999,35087,parmesan onion bake,0.564439
142531,79281,najwa s chicken soup,0.558275
230873,215178,zucchini tomato skillet,0.558262
79630,136241,eggplant tomato deluxe,0.556277
230901,38677,zucchini and caramelized onions,0.554247
124834,35130,lion knees potatoes,0.553880
42392,76222,chicken stuffed bell peppers,0.553529


In [19]:
content_precision_at_10 = evaluate_content_based(
    k=10,
    max_users=1000
)

print(
    "Content-Based Precision@10:",
    content_precision_at_10
)

Content-Based Precision@10: 0.0006006006006006007


In [20]:
def inspect_content_based_hits(max_users=20, k=10):
    users = test_ratings["user_id"].unique()[:max_users]

    results = []

    for user_id in users:
        recommendations = recommend_content_based(
            user_id=user_id,
            k=k
        )

        recommended_ids = set(recommendations["id"])

        relevant_ids = set(
            test_ratings[
                (test_ratings["user_id"] == user_id) &
                (test_ratings["rating"] >= 4)
                ]["recipe_id"]
        )

        hits = recommended_ids & relevant_ids

        results.append({
            "user_id": user_id,
            "relevant_in_test": len(relevant_ids),
            "hits": len(hits),
            "precision": len(hits) / k
        })

    return pd.DataFrame(results)

In [21]:
content_hit_check = inspect_content_based_hits(
    max_users=20,
    k=10
)

content_hit_check

,user_id,relevant_in_test,hits,precision
0,162826,87,0,0.0
1,56061,28,0,0.0
2,359220,50,0,0.0
3,47907,93,0,0.0
4,67656,175,0,0.0
5,427184,65,0,0.0
6,91392,103,0,0.0
7,196341,12,0,0.0
8,536962,13,0,0.0
9,31611,8,0,0.0


In [22]:
user_id = 162826

# Test recepti koje je korisnik dobro ocenio
user_test_relevant = test_ratings[
    (test_ratings["user_id"] == user_id) &
    (test_ratings["rating"] >= 4)
    ].copy()

# Recepti iz treninga koje je korisnik dobro ocenio
user_train_positive = train_ratings[
    (train_ratings["user_id"] == user_id) &
    (train_ratings["rating"] >= 4)
    ].copy()

print("Positive train ratings:", len(user_train_positive))
print("Positive test ratings:", len(user_test_relevant))

Positive train ratings: 321
Positive test ratings: 87


In [23]:
# Indeksi pozitivnih trening recepata
positive_indices = user_train_positive["recipe_id"].map(
    recipe_id_to_index
).dropna().astype(int).values

# Napravi profil samo od pozitivnih recepata
positive_vectors = tfidf_matrix[positive_indices]

positive_profile = positive_vectors.mean(axis=0)
positive_profile = np.asarray(positive_profile)

# Sličnost svih recepata sa pozitivnim profilom
positive_similarities = cosine_similarity(
    positive_profile,
    tfidf_matrix
).flatten()

# Proverimo koliko su test pozitivni recepti slični profilu
test_indices = user_test_relevant["recipe_id"].map(
    recipe_id_to_index
).dropna().astype(int).values

test_similarity_scores = positive_similarities[test_indices]

print("Average similarity of test-positive recipes:",
      test_similarity_scores.mean())

print("Maximum similarity of test-positive recipes:",
      test_similarity_scores.max())

Average similarity of test-positive recipes: 0.26231436468329994
Maximum similarity of test-positive recipes: 0.4318447867356791


In [24]:
diagnostic = recipes.iloc[test_indices][
    ["id", "name"]
].copy()

diagnostic["similarity_to_user_profile"] = test_similarity_scores

diagnostic.sort_values(
    "similarity_to_user_profile",
    ascending=False
).head(10)

,id,name,similarity_to_user_profile
90399,194360,garlic onion green beans,0.431845
120731,41042,leftover ham cabbage casserole,0.413280
62431,39087,creamy cajun chicken pasta,0.401125
113869,220470,jalapeno potato salad,0.395177
127011,58,low fat burgundy beef vegetable stew,0.390941
127463,26230,lucky sweet and sour sauce,0.389804
118062,150207,kittencal s delicious meaty pasta sauce,0.389099
101428,31903,ham a la king,0.385352
86201,28799,french dip,0.372695
111748,183782,irish beef stew,0.370902


In [25]:
def recommend_content_based_max_similarity(user_id, k=10):

    user_ratings = train_ratings[
        (train_ratings["user_id"] == user_id) &
        (train_ratings["rating"] >= 4)
        ].copy()

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Samo recepti koji postoje u TF-IDF matrici
    user_ratings = user_ratings[
        user_ratings["recipe_id"].isin(recipe_id_to_index.index)
    ]

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Indeksi pozitivno ocenjenih recepata
    positive_indices = user_ratings["recipe_id"].map(
        recipe_id_to_index
    ).values

    positive_vectors = tfidf_matrix[positive_indices]

    # Računamo sličnost u batch-evima da ne zauzmemo previše RAM-a
    batch_size = 10000
    max_scores = np.full(tfidf_matrix.shape[0], -1.0)

    for start in range(0, tfidf_matrix.shape[0], batch_size):
        end = min(start + batch_size, tfidf_matrix.shape[0])

        batch = tfidf_matrix[start:end]

        similarities = cosine_similarity(
            batch,
            positive_vectors
        )

        max_scores[start:end] = similarities.max(axis=1)

    # Ne preporučujemo već ocenjene recepte
    rated_recipe_ids = user_ratings["recipe_id"].values

    rated_indices = [
        recipe_id_to_index[recipe_id]
        for recipe_id in rated_recipe_ids
    ]

    max_scores[rated_indices] = -1

    # Top-k
    top_indices = np.argsort(max_scores)[-k:][::-1]

    recommendations = recipes.iloc[top_indices][
        ["id", "name"]
    ].copy()

    recommendations["score"] = max_scores[top_indices]

    return recommendations

In [26]:
content_max_recommendations = recommend_content_based_max_similarity(
    user_id=162826,
    k=10
)

content_max_recommendations

,id,name,score
59049,391137,country club chicken,0.819903
142223,356220,my reuben sandwich with sauerkraut,0.780027
130037,59280,maple glazed roasted carrots,0.774745
207745,240825,taco cornbread pizza,0.753040
150481,304943,out of this world cake,0.751612
179278,218467,salmon with lime and cilantro sauce,0.750646
172189,335329,red lobster tartar sauce by todd wilbur,0.744346
104678,119573,herb three cheese omelet,0.734835
143163,383595,nestle toll house pie,0.732719
46033,170878,chicken that melts in your mouth,0.729286


In [27]:
def evaluate_content_based_max_similarity(k=10, max_users=1000):
    users = test_ratings["user_id"].unique()[:max_users]

    precisions = []

    for user_id in users:

        recommendations = recommend_content_based_max_similarity(
            user_id=user_id,
            k=k
        )

        if len(recommendations) == 0:
            continue

        recommended_ids = set(recommendations["id"])

        relevant_ids = set(
            test_ratings[
                (test_ratings["user_id"] == user_id) &
                (test_ratings["rating"] >= 4)
                ]["recipe_id"]
        )

        if len(relevant_ids) == 0:
            continue

        hits = len(recommended_ids & relevant_ids)

        precisions.append(hits / k)

    return np.mean(precisions)

In [ ]:
content_max_precision_at_10 = evaluate_content_based_max_similarity(
    k=10,
    max_users=1000
)

print(
    "Content-Based Max Similarity Precision@10:",
    content_max_precision_at_10
)

In [29]:
content_max_precision_at_10 = evaluate_content_based_max_similarity(
    k=10,
    max_users=100
)

print(
    "Content-Based Max Similarity Precision@10:",
    content_max_precision_at_10
)

KeyboardInterrupt: 

In [30]:
def evaluate_content_based_fast(k=10, max_users=1000, batch_size=10):
    users = test_ratings["user_id"].unique()[:max_users]

    precisions = []

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]

        profiles = []
        valid_users = []

        for user_id in batch_users:

            user_ratings = train_ratings[
                (train_ratings["user_id"] == user_id) &
                (train_ratings["rating"] >= 4)
                ]

            user_ratings = user_ratings[
                user_ratings["recipe_id"].isin(
                    recipe_id_to_index.index
                )
            ]

            if len(user_ratings) == 0:
                continue

            indices = user_ratings["recipe_id"].map(
                recipe_id_to_index
            ).values

            vectors = tfidf_matrix[indices]

            weights = user_ratings["rating"].values.astype(float)

            profile = vectors.multiply(
                weights.reshape(-1, 1)
            ).sum(axis=0)

            profile = profile / weights.sum()

            profiles.append(profile)
            valid_users.append(user_id)

        if len(profiles) == 0:
            continue

        profiles = np.vstack([
            np.asarray(profile).ravel()
            for profile in profiles
        ])

        # Sličnost svih profila iz batch-a sa svim receptima
        similarities = cosine_similarity(
            profiles,
            tfidf_matrix
        )

        for i, user_id in enumerate(valid_users):

            scores = similarities[i].copy()

            # Ne preporučujemo recepte koje je korisnik već ocenio
            train_user_recipe_ids = train_ratings[
                train_ratings["user_id"] == user_id
                ]["recipe_id"].values

            rated_indices = [
                recipe_id_to_index[recipe_id]
                for recipe_id in train_user_recipe_ids
                if recipe_id in recipe_id_to_index.index
            ]

            scores[rated_indices] = -1

            top_indices = np.argpartition(
                scores,
                -k
            )[-k:]

            recommended_ids = set(
                recipes.iloc[top_indices]["id"]
            )

            relevant_ids = set(
                test_ratings[
                    (test_ratings["user_id"] == user_id) &
                    (test_ratings["rating"] >= 4)
                    ]["recipe_id"]
            )

            if len(relevant_ids) == 0:
                continue

            hits = len(
                recommended_ids & relevant_ids
            )

            precisions.append(hits / k)

        print(
            f"Processed {min(start + batch_size, len(users))}/{len(users)} users"
        )

    return np.mean(precisions)

In [31]:
content_precision_at_10_fast = evaluate_content_based_fast(
    k=10,
    max_users=100,
    batch_size=10
)

print(
    "Content-Based Precision@10:",
    content_precision_at_10_fast
)

Processed 10/100 users
Processed 20/100 users
Processed 30/100 users
Processed 40/100 users
Processed 50/100 users
Processed 60/100 users
Processed 70/100 users
Processed 80/100 users
Processed 90/100 users
Processed 100/100 users
Content-Based Precision@10: 0.001


In [32]:
def evaluate_content_based_recall_fast(k=10, max_users=100, batch_size=10):
    users = test_ratings["user_id"].unique()[:max_users]

    recalls = []

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]

        profiles = []
        valid_users = []

        for user_id in batch_users:

            user_ratings = train_ratings[
                (train_ratings["user_id"] == user_id) &
                (train_ratings["rating"] >= 4)
                ]

            user_ratings = user_ratings[
                user_ratings["recipe_id"].isin(
                    recipe_id_to_index.index
                )
            ]

            if len(user_ratings) == 0:
                continue

            indices = user_ratings["recipe_id"].map(
                recipe_id_to_index
            ).values

            vectors = tfidf_matrix[indices]

            weights = user_ratings["rating"].values.astype(float)

            profile = vectors.multiply(
                weights.reshape(-1, 1)
            ).sum(axis=0)

            profile = profile / weights.sum()

            profiles.append(profile)
            valid_users.append(user_id)

        if len(profiles) == 0:
            continue

        profiles = np.vstack([
            np.asarray(profile).ravel()
            for profile in profiles
        ])

        similarities = cosine_similarity(
            profiles,
            tfidf_matrix
        )

        for i, user_id in enumerate(valid_users):

            scores = similarities[i].copy()

            train_user_recipe_ids = train_ratings[
                train_ratings["user_id"] == user_id
                ]["recipe_id"].values

            rated_indices = [
                recipe_id_to_index[recipe_id]
                for recipe_id in train_user_recipe_ids
                if recipe_id in recipe_id_to_index.index
            ]

            scores[rated_indices] = -1

            top_indices = np.argpartition(
                scores,
                -k
            )[-k:]

            recommended_ids = set(
                recipes.iloc[top_indices]["id"]
            )

            relevant_ids = set(
                test_ratings[
                    (test_ratings["user_id"] == user_id) &
                    (test_ratings["rating"] >= 4)
                    ]["recipe_id"]
            )

            if len(relevant_ids) == 0:
                continue

            hits = len(
                recommended_ids & relevant_ids
            )

            recalls.append(
                hits / len(relevant_ids)
            )

        print(
            f"Processed {min(start + batch_size, len(users))}/{len(users)} users"
        )

    return np.mean(recalls)

In [33]:
content_recall_at_10 = evaluate_content_based_recall_fast(
    k=10,
    max_users=100,
    batch_size=10
)

print(
    "Content-Based Recall@10:",
    content_recall_at_10
)

Processed 10/100 users
Processed 20/100 users
Processed 30/100 users
Processed 40/100 users
Processed 50/100 users
Processed 60/100 users
Processed 70/100 users
Processed 80/100 users
Processed 90/100 users
Processed 100/100 users
Content-Based Recall@10: 4.830917874396135e-05
